In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [10]:
from app.config import settings

In [11]:
model_name = settings.EMBEDDING_MODEL
dimension = settings.EMBEDDING_DIMENSION
client = GoogleGenerativeAIEmbeddings(
            model=model_name,
            google_api_key=settings.GEMINI_API_KEY,
        )

In [12]:
client.embed_query("hi how are you?")

[-0.023745231330394745,
 0.0083352355286479,
 0.004480163101106882,
 -0.08734215050935745,
 -0.005805687978863716,
 -0.004170320928096771,
 -0.010113386437296867,
 0.0020749305840581656,
 0.009593961760401726,
 0.007590307854115963,
 -0.00020999573462177068,
 -0.011602381244301796,
 0.003844328923150897,
 0.008087096735835075,
 0.1572096049785614,
 -0.007285694591701031,
 0.007966679520905018,
 0.0067625935189425945,
 0.011660374701023102,
 -0.02323327399790287,
 0.002747324062511325,
 -0.000856351456604898,
 -0.0038450937718153,
 0.0055117253214120865,
 -0.004431517329066992,
 0.01339320745319128,
 0.011473800987005234,
 -0.0029700114391744137,
 0.014682948589324951,
 -0.004070257768034935,
 0.008310345932841301,
 -0.003404349321499467,
 -0.0012621929636225104,
 0.013595560565590858,
 0.0074196248315274715,
 0.005093727260828018,
 0.009267681278288364,
 0.01972142420709133,
 -0.00875735655426979,
 0.003038765164092183,
 0.012902314774692059,
 0.00021392149210441858,
 -0.01437191013246

In [2]:
from app.ingestion.processor import IngestionProcessor

d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ingest = IngestionProcessor()

In [1]:
import socket
import os

host = "generativelanguage.googleapis.com"

try:
    print("DNS:", socket.getaddrinfo(host, 443)[0][4])
except OSError as e:
    print("DNS failed:", repr(e))

print("Proxy variables:", {
    name: os.environ.get(name)
    for name in ("HTTPS_PROXY", "HTTP_PROXY", "ALL_PROXY", "NO_PROXY")
})


DNS: ('172.217.112.4', 443)
Proxy variables: {'HTTPS_PROXY': None, 'HTTP_PROXY': None, 'ALL_PROXY': None, 'NO_PROXY': None}


In [28]:
vector = client.embed_query("سلام")
print("Embedding OK — dimensions:", len(vector))

Embedding OK — dimensions: 3072


In [26]:
from pathlib import Path

ingest.ingest_file(
    Path("DATA/true_data/parallel_work_queue.txt"),
    source_type="txt",
)

ResponseHandlingException: [WinError 10061] No connection could be made because the target machine actively refused it

In [27]:
import traceback

try:
    ingest.ingest_file(
        Path("DATA/true_data/parallel_work_queue.txt"),
        source_type="txt",
    )
except Exception as e:
    print("Exception:", type(e).__name__, str(e))
    print("\nآخرین فراخوانی‌های traceback:")
    for frame in traceback.extract_tb(e.__traceback__)[-12:]:
        print(f"{frame.filename}:{frame.lineno} — {frame.name}")


Exception: ResponseHandlingException [WinError 10061] No connection could be made because the target machine actively refused it

آخرین فراخوانی‌های traceback:
d:\Projects\RAG_Systems\DriveRAG\app\ingestion\processor.py:115 — _ensure_collection
d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\qdrant_client\qdrant_client.py:1609 — collection_exists
d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:1935 — collection_exists
d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\qdrant_client\http\api\collections_api.py:438 — collection_exists
d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\qdrant_client\http\api\collections_api.py:67 — _build_for_collection_exists
d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\qdrant_client\http\api_client.py:98 — request
d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\qdrant_client\http\api_client.py:115 — send
d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\qdrant_clien

In [29]:
try:
    print("Qdrant reachable:", ingest.qdrant_client.get_collections())
except Exception as e:
    print(type(e).__name__, str(e))

ResponseHandlingException [WinError 10061] No connection could be made because the target machine actively refused it


In [30]:
print(settings.QDRANT_CLUSTER_ENDPOINT)

https://e2c3824c-e709-4f36-8e77-7e6382343038.europe-west3-0.gcp.cloud.qdrant.io


In [4]:
from urllib.parse import urlparse
from app.config import settings

endpoint = str(settings.QDRANT_CLUSTER_ENDPOINT).strip()
parsed = urlparse(endpoint)

print("Scheme:", parsed.scheme)
print("Host:", parsed.hostname)
print("Port:", parsed.port)
print("Collection:", settings.QDRANT_COLLECTION)


Scheme: https
Host: e2c3824c-e709-4f36-8e77-7e6382343038.europe-west3-0.gcp.cloud.qdrant.io
Port: None
Collection: DriveRAG


In [5]:
import socket
from urllib.parse import urlparse
from app.config import settings

host = urlparse(str(settings.QDRANT_CLUSTER_ENDPOINT).strip()).hostname

print("Host present:", bool(host))
if host:
    try:
        with socket.create_connection((host, 443), timeout=10):
            print("TCP 443: connected")
    except OSError as e:
        print("TCP 443:", type(e).__name__, str(e))


Host present: True
TCP 443: connected


In [6]:
from qdrant_client import QdrantClient
from app.config import settings

client = QdrantClient(
    url=str(settings.QDRANT_CLUSTER_ENDPOINT).strip(),
    api_key=settings.QDRANT_API_KEY,  # اگر نام تنظیمات کلید در پروژه‌ات متفاوت است، اصلاحش کن
    prefer_grpc=False,
    timeout=15,
)

try:
    print(client.get_collections())
except Exception as e:
    print(type(e).__name__, str(e))


collections=[CollectionDescription(name='DriveRAG')]


In [4]:
from pathlib import Path
from app.ingestion.processor import IngestionProcessor

ingest = IngestionProcessor()
file_path = Path("DATA/true_data/pods_autoscale.html")

ingest.ingest_file(file_path, source_type="true_data")


10:19:48.865 Processing File
10:19:48.866   HTML Parsing
10:19:48.902   ✂️ Text Chunking (LangChain)
10:19:48.912     ✅ Generated 28 chunks using LangChain
10:19:48.918   Saved processed data to D:\Projects\RAG_Systems\DriveRAG\data\processed\true_data\pods_autoscale.html.json
10:19:48.919   Vectorizing & Indexing
10:20:21.889     Indexed pods_autoscale.html: chunks 0..4
10:20:43.125     Indexed pods_autoscale.html: chunks 5..9
10:21:05.431     Indexed pods_autoscale.html: chunks 10..14
10:21:28.222     Indexed pods_autoscale.html: chunks 15..19
10:21:50.215     Indexed pods_autoscale.html: chunks 20..24
10:21:55.730     Indexed pods_autoscale.html: chunks 25..27
10:21:55.731   Successfully indexed pods_autoscale.html


In [1]:
import logfire

logfire.configure()
logfire.info("DriveRAG Logfire test")
logfire.force_flush()

Logfire project URL: https://logfire-us.pydantic.dev/mhmdrza9789/drive-rag

10:04:59.542 DriveRAG Logfire test


False

In [7]:
import os
from urllib.parse import urlparse
import socket

for name in ("QDRANT_URL", "HTTPS_PROXY", "HTTP_PROXY", "ALL_PROXY"):
    value = os.getenv(name)
    if not value:
        continue

    host = urlparse(value).hostname
    if not host:
        print(f"{name}: set, but hostname could not be parsed")
        continue

    try:
        ip = socket.getaddrinfo(host, None)[0][4][0]
        print(f"{name}: {host} -> {ip}")
    except socket.gaierror as exc:
        print(f"{name}: {host} -> DNS FAILED ({exc})")


In [9]:
import socket
from urllib.parse import urlparse
from app.config import settings

endpoint = str(settings.QDRANT_CLUSTER_ENDPOINT).strip()
host = urlparse(endpoint).hostname

if not host:
    print("Endpoint hostname: NOT PARSED")
else:
    try:
        socket.getaddrinfo(host, None)
        print("Endpoint DNS: OK")
    except socket.gaierror as exc:
        print(f"Endpoint DNS: FAILED ({exc})")


Endpoint DNS: OK


In [10]:
import traceback

try:
    collections = ingest.qdrant_client.get_collections()
    print("Qdrant API: OK")
except Exception as exc:
    print("Qdrant API:", type(exc).__name__, str(exc))
    print("Root cause:")
    root = exc
    while root.__cause__ is not None:
        root = root.__cause__
    print(type(root).__name__, str(root))


Qdrant API: ResponseHandlingException _ssl.c:1011: The handshake operation timed out
Root cause:
ResponseHandlingException _ssl.c:1011: The handshake operation timed out


In [5]:
from pathlib import Path
from app.ingestion.processor import IngestionProcessor

root = Path("DATA/true_data")
print("Folder exists:", root.is_dir())

ingest = IngestionProcessor()
ingest.ingest_directory(root, source_type="true_data")


Folder exists: True
10:27:31.959 Ingest Directory
10:27:31.960   Processing File
10:27:32.330     ✂️ Text Chunking (LangChain)
10:27:32.331       ✅ Generated 27 chunks using LangChain
10:27:32.333     Saved processed data to D:\Projects\RAG_Systems\DriveRAG\data\processed\true_data\architecture.pptx.json
10:27:32.334     Vectorizing & Indexing
10:27:40.383       Indexed architecture.pptx: chunks 0..4
10:27:46.811       Indexed architecture.pptx: chunks 5..9
10:27:53.214       Indexed architecture.pptx: chunks 10..14
10:27:59.580       Indexed architecture.pptx: chunks 15..19
10:28:05.957       Indexed architecture.pptx: chunks 20..24
10:28:07.003       Indexed architecture.pptx: chunks 25..26
10:28:07.004     Successfully indexed architecture.pptx
10:28:07.005   Processing File
10:28:07.369     ✂️ Text Chunking (LangChain)
10:28:07.372       ✅ Generated 54 chunks using LangChain
10:28:07.376     Saved processed data to D:\Projects\RAG_Systems\DriveRAG\data\processed\true_data\cronjobs.

In [6]:
from pathlib import Path
from app.ingestion.processor import IngestionProcessor

root = Path("DATA/noisy_data")
print("Folder exists:", root.is_dir())

ingest = IngestionProcessor()
ingest.ingest_directory(root, source_type="noisy_data")

Folder exists: True


Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 201 0 (offset 0)


10:40:28.420 Ingest Directory
10:40:28.460   Processing File
10:40:28.482     PDF Parsing (local)
10:40:29.738       PDF text extracted
10:40:29.740     ✂️ Text Chunking (LangChain)
10:40:29.741       ✅ Generated 15 chunks using LangChain
10:40:29.744     Saved processed data to D:\Projects\RAG_Systems\DriveRAG\data\processed\noisy_data\100... Technology Overview - Slides - Toronto (August 2016).pdf.json
10:40:29.745     Vectorizing & Indexing
10:40:37.733       Indexed 100G Networking Technology Overview - Slides - Toronto (August 2016).pdf: chunks 0..4
10:40:45.218       Indexed 100G Networking Technology Overview - Slides - Toronto (August 2016).pdf: chunks 5..9
10:40:53.476       Indexed 100G Networking Technology Overview - Slides - Toronto (August 2016).pdf: chunks 10..14
10:40:53.478     Successfully indexed 100G Networking Technology Overview - Slides - Toronto (August 2016).pdf
10:40:53.481   Processing File
10:40:53.481     PDF Parsing (local)
10:41:00.605       PDF text extr

Embedding request failed; retrying in 1.6 seconds (1/5): Error embedding content: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.


10:51:34.247       Indexed A Brief Introduction to Neural Networks (neuronalenetze-en-zeta2-2col-dkrieselcom).pdf: chunks 30..34
10:51:42.817       Indexed A Brief Introduction to Neural Networks (neuronalenetze-en-zeta2-2col-dkrieselcom).pdf: chunks 35..39
10:51:50.269       Indexed A Brief Introduction to Neural Networks (neuronalenetze-en-zeta2-2col-dkrieselcom).pdf: chunks 40..44
10:51:57.587       Indexed A Brief Introduction to Neural Networks (neuronalenetze-en-zeta2-2col-dkrieselcom).pdf: chunks 45..49
10:52:04.820       Indexed A Brief Introduction to Neural Networks (neuronalenetze-en-zeta2-2col-dkrieselcom).pdf: chunks 50..54
10:52:12.252       Indexed A Brief Introduction to Neural Networks (neuronalenetze-en-zeta2-2col-dkrieselcom).pdf: chunks 55..59
10:52:19.267       Indexed A Brief Introduction to Neural Networks (neuronalenetze-en-zeta2-2col-dkrieselcom).pdf: chunks 60..64
10:52:26.582       Indexed A Brief Introduction to Neural Networks (neuronalenetze-en-zeta2-2col-

...

11:37:08.961     Failed to process A Trip Through The Graphics Pipeline - All (2011).docx: Error embedding content: 429 You exceeded your current quota, ...l"
  }
  quota_value: 1000
}
, retry_delay {
  seconds: 51
}
]

FAILED: DATA\noisy_data\A Trip Through The Graphics Pipeline - All (2011).docx
11:37:08.978   Processing File
11:37:08.979     Failed to process gitignore: Unsupported file type: DATA\noisy_data\gitignore

FAILED: DATA\noisy_data\gitignore
11:37:09.025   Processing File
11:37:09.027     Text Parsing
11:37:09.054     ✂️ Text Chunking (LangChain)
11:37:09.055       ✅ Generated 1 chunks using LangChain
11:37:09.058     Saved processed data to D:\Projects\RAG_Systems\DriveRAG\data\processed\noisy_data\README.txt.json
11:37:09.059     Vectorizing & Indexing


Traceback (most recent call last):
  File "d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\google\api_core\grpc_helpers.py", line 76, in error_remapped_callable
    return callable_(*args, **kwargs)
  File "d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\grpc\_interceptor.py", line 276, in __call__
    response, ignored_call = self._with_call(
                             ~~~~~~~~~~~~~~~^
        request,
        ^^^^^^^^
    ...<4 lines>...
        compression=compression,
        ^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\grpc\_interceptor.py", line 331, in _with_call
    return call.result(), call
           ~~~~~~~~~~~^^
  File "d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\grpc\_channel.py", line 447, in result
    raise self
  File "d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\grpc\_interceptor.py", line 314, in continuation
    response, call = self._thunk(new_method).with_call(
 

11:37:45.078     Failed to process README.txt: Error embedding content: 429 You exceeded your current quota, ...l"
  }
  quota_value: 1000
}
, retry_delay {
  seconds: 15
}
]

FAILED: DATA\noisy_data\README.txt


Traceback (most recent call last):
  File "d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\google\api_core\grpc_helpers.py", line 76, in error_remapped_callable
    return callable_(*args, **kwargs)
  File "d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\grpc\_interceptor.py", line 276, in __call__
    response, ignored_call = self._with_call(
                             ~~~~~~~~~~~~~~~^
        request,
        ^^^^^^^^
    ...<4 lines>...
        compression=compression,
        ^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\grpc\_interceptor.py", line 331, in _with_call
    return call.result(), call
           ~~~~~~~~~~~^^
  File "d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\grpc\_channel.py", line 447, in result
    raise self
  File "d:\Projects\RAG_Systems\DriveRAG\.venv\Lib\site-packages\grpc\_interceptor.py", line 314, in continuation
    response, call = self._thunk(new_method).with_call(
 

RuntimeError: Ingestion failed for 50 file(s): DATA\noisy_data\A Brief Introduction to Neural Networks (neuronalenetze-en-zeta2-2col-dkrieselcom).pdf, DATA\noisy_data\A Brief Introduction to the Standard Annotation Language (SAL) - 2006.pdf, DATA\noisy_data\A Brief Tutorial on Database Queries, Data Mining, and OLAP (hamel-197-manuscript-final).pdf, DATA\noisy_data\A Case Study in Optimizing HTM-Enabled Dynamic Data Structures - Patricia Tries (2015).pdf, DATA\noisy_data\A Catalogue of Optimizing Transformations (1971-allen-catalog).pdf, DATA\noisy_data\A Comparison of Programming Languages in Economics (16-Jun-2014).pdf, DATA\noisy_data\A Comparison of Software and Hardware Techniques for x86 Virtualization - Oct 2015 (asplos235_adams).pdf, DATA\noisy_data\A Forensic Analysis of CSG 11 Encounter with an Anomalous Aerial Vehicle.pdf, DATA\noisy_data\A Framework for Building Extensible C++ Class Libraries - 1993.pdf, DATA\noisy_data\A GPU-friendly Skiplist Algorith - 2014 (GPUSkiplist).